# Агент "Куда сходить в Алматы"

ТЗ #2: парсинг sxodim.com/almaty + агент, отвечающий на вопросы о том, куда сходить.

## Раздел 1: Парсинг сайта

Скрейпинг запускается один раз и кэшируется в `sxodim_raw.json` — при повторном запуске ноутбука сеть не дёргается, если файл уже существует. Логика парсинга живёт в `scraper.py` (см. рядом), ноутбук только вызывает её и показывает результат.

**Объём данных — сознательное решение, не случайный обрыв.** События (`/almaty/afisha?page=1..9`) берутся полностью — сайт отдаёт все 139 событий за 9 страниц. Места устроены иначе: общий листинг `/almaty/places` — это 119 страниц (~2260 мест), сильно перекошенных по категориям (одни только кафе — 77 страниц). Чтобы не терять маленькие, но важные для вопросов ТЗ категории (например, `dlja-detej` — «для детей» — всего 2 страницы) в шуме больших, мы берём **первые 2 страницы каждой из 22 категорий мест** (`places/<category>?page=1,2`), а не первые N страниц общего листинга. Это даёт сбалансированный срез по всем типам мест вместо перекоса в сторону кафе/ресторанов — см. `docs/superpowers/specs/2026-09-28-sxodim-agent-design.md`, раздел 4.1.

In [1]:
import json
from pathlib import Path

import requests

import scraper

RAW_PATH = Path("sxodim_raw.json")

In [2]:
if RAW_PATH.exists():
    records = json.loads(RAW_PATH.read_text(encoding="utf-8"))
    print(f"Loaded {len(records)} cached records from {RAW_PATH}")
else:
    session = requests.Session()
    fetch = scraper.make_fetcher(session)
    records = scraper.scrape_all(fetch)
    scraper.save_raw(records, str(RAW_PATH))
    print(f"Scraped {len(records)} records -> {RAW_PATH}")

Loaded 592 cached records from sxodim_raw.json


In [3]:
events = [r for r in records if r["type"] == "event"]
places = [r for r in records if r["type"] == "place"]
print(f"events: {len(events)}, places: {len(places)}, total: {len(records)}")

events: 136, places: 456, total: 592


In [4]:
print("Пример события:")
print(json.dumps(events[0], ensure_ascii=False, indent=2))
print()
print("Пример места:")
print(json.dumps(places[0], ensure_ascii=False, indent=2))

Пример события:
{
  "type": "event",
  "url": "https://sxodim.com/almaty/event/abzal-teshovty-koncerti",
  "name": "Абзал Утешов Алматыда",
  "category": "Концерты во Дворце Республики",
  "description": "Абзал Утешов – қазақстандық эстраданың жас әрі жарқын өкілдерінің бірі. Ол тыңдармандардың жүрегінен орын алған әсерлі әндері мен шынайы орындау мәнері арқылы кеңінен танымал болды.",
  "address": "Республика сарайы, Достық даңғылы, 56",
  "date_start": "2026-10-07T20:00:00+05:00",
  "price": 11000,
  "currency": "KZT",
  "rating": null,
  "image": "https://sxodim.com/uploads/posts/2026/08/02/optimized/d0fcc3bb75454a3f7c4f79e31ae74adb_1522x570-q-85.jpg"
}

Пример места:
{
  "type": "place",
  "url": "https://sxodim.com/almaty/place/17st-lounge-bar",
  "name": "17st. Lounge Bar",
  "category": "Бары",
  "description": "Посетители смогут окунуться в атмосферу вкусных стейков, крафтового пенного и дымных сетов.",
  "address": "17st. Lounge Bar, ул. Розыбакиева, 166",
  "date_start": null

## Раздел 2: Структурирование данных

Дальше — сырые записи из `sxodim_raw.json` превращаются в финальный `sxodim_data.json`: каждая запись получает стабильный `id` и список `tags`. Теги выводятся **детерминированно** из категории (подстрочные правила вида "категория содержит «концерт»" → тег `концерт`) и из цены (`price == 0` → `бесплатно`) — без обращения к LLM. Логика — в `normalize.py`.

In [5]:
import normalize

data = normalize.normalize_all(records)
normalize.save_data(data, "sxodim_data.json")
print(f"Normalized {len(data)} records -> sxodim_data.json")

Normalized 592 records -> sxodim_data.json


In [6]:
import collections

tag_counts = collections.Counter(t for r in data for t in r["tags"])
no_tags = sum(1 for r in data if not r["tags"])
print(f"tag distribution: {tag_counts.most_common()}")
print(f"records with no tags: {no_tags} (categories: {sorted({r['category'] for r in data if not r['tags']})})")

tag distribution: [('активный_отдых', 193), ('искусство', 92), ('еда', 90), ('ночная_жизнь', 78), ('образование', 42), ('концерт', 33), ('для_детей', 19), ('романтика', 11)]
records with no tags: 49 (categories: ['Коворкинги', 'Красота и здоровье', 'Отели'])


In [7]:
sample_event = next(r for r in data if r["type"] == "event")
sample_place = next(r for r in data if r["type"] == "place")
print("Структурированное событие:")
print(json.dumps(sample_event, ensure_ascii=False, indent=2))
print()
print("Структурированное место:")
print(json.dumps(sample_place, ensure_ascii=False, indent=2))

Структурированное событие:
{
  "type": "event",
  "url": "https://sxodim.com/almaty/event/abzal-teshovty-koncerti",
  "name": "Абзал Утешов Алматыда",
  "category": "Концерты во Дворце Республики",
  "description": "Абзал Утешов – қазақстандық эстраданың жас әрі жарқын өкілдерінің бірі. Ол тыңдармандардың жүрегінен орын алған әсерлі әндері мен шынайы орындау мәнері арқылы кеңінен танымал болды.",
  "address": "Республика сарайы, Достық даңғылы, 56",
  "date_start": "2026-10-07T20:00:00+05:00",
  "price": 11000,
  "currency": "KZT",
  "rating": null,
  "image": "https://sxodim.com/uploads/posts/2026/08/02/optimized/d0fcc3bb75454a3f7c4f79e31ae74adb_1522x570-q-85.jpg",
  "id": "event-abzal-teshovty-koncerti",
  "tags": [
    "концерт"
  ]
}

Структурированное место:
{
  "type": "place",
  "url": "https://sxodim.com/almaty/place/17st-lounge-bar",
  "name": "17st. Lounge Bar",
  "category": "Бары",
  "description": "Посетители смогут окунуться в атмосферу вкусных стейков, крафтового пенного

## Раздел 3: Агент — отвечает на вопросы

RAG-lite без векторной БД: LLM (Ollama, `qwen2.5-7b-instruct`) переводит вопрос в JSON-фильтр (`parse_intent`), чистый Python фильтрует `sxodim_data.json` по дате/цене/тегам (`filter_candidates`), затем LLM формирует финальный ответ **только** по отобранным кандидатам (`generate_answer`). Код — в `agent.py`, больше примеров — в `agent_examples.md`.

In [8]:
import datetime

import agent

chat = agent.make_ollama_chat()
now = datetime.datetime.now().astimezone()

questions = [
    "Куда сходить на выходных?",
    "Что нового открылось в Алматы?",
    "Посоветуй место для свидания",
    "Куда сводить ребенка?",
    "Какие концерты будут?",
    "Где вкусно поесть?",
]

In [9]:
for q in questions:
    intent = agent.parse_intent(q, chat)
    candidates = agent.filter_candidates(data, intent, now=now)
    answer = agent.generate_answer(q, candidates, chat)
    print(f"Q: {q}")
    print(f"intent: {intent}")
    print(f"A: {answer}")
    print("-" * 80)

Q: Куда сходить на выходных?
intent: {'date_filter': 'weekend', 'intent_tags': [], 'price_max': None}
A: Конечно, на выходных у вас есть несколько вариантов, чтобы провести время с комфортом и весельем!

1. **City Pop Stories: Tokyo Nights — Юлия Яковлева и резиденты EverJazz**
   - **Адрес**: Джаз-клуб EverJazz, ул. Жандосова, 1/1
   - **Дата и время**: 3 октября, с 19:00 до 22:00
   - **Цена**: 4000 KZT

2. **Тройной стендап. Новые шутки (by Stand Up Camp)**
   - **Адрес**: Coffee House Grain, ул. Курмангазы, 97
   - **Дата и время**: 3 октября, с 20:00 до 23:00

3. **Ночь стендапа (3 октября)**
   - **Адрес**: Punch Stand Up Club, ул. Каирбекова, 35А
   - **Дата и время**: 3 октября, с 23:30
   - **Цена**: 2000 KZT

Это отличные варианты для веселого и развлекательного времяпрепровождения!
--------------------------------------------------------------------------------


Q: Что нового открылось в Алматы?
intent: {'date_filter': 'any', 'intent_tags': [], 'price_max': None}
A: Конечно, вот несколько вариантов, которые могут вам понравиться:

1. **17st. Lounge Bar** — это уютное место для вкусных стейков и крафтового пенного. Адрес: ул. Розыбакиева, 166. Несмотря на то что дата открытия не указана, это отличное место для приятного вечера.

2. **Алматинский Арбат** — идеальное место для прогулки и развлечений. Здесь всегда кипит жизнь, выступают уличные артисты и чувствуется расслабляющая атмосфера. Адрес: пр. Жибек Жолы, 99.

3. **Аквапарк «На Море»** — семейный комплекс с множеством водных развлечений для всей семьи. Адрес: г. Конаев, ул. Северное побережье, 68. Удобно, если вы ищете активный отдых с детьми.

4. **Аквапарк «Восьмое чудо света»** — развлекательный комплекс с водными горками и бассейнами, расположенный в ЦПКиО. Адрес: ЦПКиО им. Горького, ул. Гоголя, 1. Удобный вариант для прогулки и отдыха.

Надеюсь, вам понравится один из этих вариантов!


Q: Посоветуй место для свидания
intent: {'date_filter': 'any', 'intent_tags': ['романтика'], 'price_max': None}
A: Конечно, предлагаю два варианта для романтического вечера:

1. **AsiaThaiSpa**
   - **Адрес:** мкр. Самал-3, 25
   - **Описание:** SPA-центр, предлагающий 25 видов массажа и SPA-программ от 30 минут до 3 часов.
   - **Теги:** романтика

2. **Morning Spa**
   - **Адрес:** улица Зейна Шашкина, 11
   - **Описание:** Уникальный спа-салон с более 15 видами массажа из разных стран, включая тибетский, турецкий, монгольский, тайский и балийский массажи.
   - **Теги:** романтика
--------------------------------------------------------------------------------


Q: Куда сводить ребенка?
intent: {'date_filter': 'any', 'intent_tags': ['для_детей'], 'price_max': None}
A: Конечно, предлагаю два варианта:

1. **Парк развлечений «Аватария» (парк «1001 ночь»)** — это яркий и веселый парк, где дети могут насладиться аттракционами и играми. Адрес: мкр. Жетысу-4, 32/2, ТД «Керуен» 1-2 этаж. Рейтинг: 5.

2. **Консультация детского психолога от центра ZERBALA** — центр приглашает родителей на консультацию детского психолога, где можно получить информацию о трудностях в поведении ребенка. Адрес: Центр Интеллектуального Развития Детей ZERBALA, 2-й мкр, 20Б. Рейтинг: 5.
--------------------------------------------------------------------------------


Q: Какие концерты будут?
intent: {'date_filter': 'any', 'intent_tags': ['концерт'], 'price_max': None}
A: Конечно, вот несколько предложений с концертами:

1. **Абзал Утешов Алматыда**
   - **Дата:** 7 октября 2026 года, время начала 20:00
   - **Адрес:** Республика сарайы, Достық даңғылы, 56
   - **Цена:** 11 000 KZT

2. **All stars jam session – Джазовая музыка и импровизация**
   - **Дата:** 5 октября 2026 года, время начала 20:00
   - **Адрес:** Джаз-клуб EverJazz, ул. Жандосова, 1/1
   - **Цена:** 2 000 KZT

3. **City Pop Stories: Tokyo Nights — Юлия Яковлева и резиденты EverJazz**
   - **Дата:** 3 октября 2026 года, время начала 19:00
   - **Адрес:** Джаз-клуб EverJazz, ул. Жандосова, 1/1
   - **Цена:** 4 000 KZT

4. **Dair Ard – Кельтский фолк**
   - **Дата:** 2 октября 2026 года, время начала 22:00
   - **Адрес:** Джаз-клуб EverJazz, ул. Жандосова, 1/1
   - **Цена:** 4 000 KZT

Надеюсь, вам понравится один из этих вариантов! Если что-то не подходит, дайте знать, могу предложить

Q: Где вкусно поесть?
intent: {'date_filter': 'any', 'intent_tags': ['еда'], 'price_max': None}
A: Я бы рекомендовал два отличных места, где можно вкусно поесть:

1. **Анticaфе «Как дома»** - отличное место, где вы сможете не только вкусно поесть, но и провести время с друзьями или семьей. Удобное расположение на улице Байзакова 223. Рейтинг — 5.

2. **Coffee Original на Абая** - здесь можно насладиться сытым завтраком и свежезаваренным кофе. Адрес — пр. Абая, 95. Рейтинг — 5.

Оба места имеют высокие оценки и создают уютную атмосферу для отдыха и общения.
--------------------------------------------------------------------------------


Больше примеров (включая вопросы с ценой и датой) — в `agent_examples.md`. Дальше (опционально, следующий этап): ORPO-дообучение для более дружелюбного тона ответов.